# Sora — Improved Training on Google Colab

This notebook extracts the project ZIP, prepares the reviewed dataset, fine-tunes Qwen2.5-0.5B-Instruct with assistant-only loss, runs a fixed prompt check, and launches Gradio. Select a GPU runtime first. The ZIP contains scripts/data, not trained weights.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose sora_model_improved.zip
zip_path = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Uploaded:', zip_path)

In [ ]:
!unzip -o "{zip_path}" -d /content/
%cd /content/Sora

## 1. Install requirements

This avoids explicitly upgrading Colab's already-imported IPython stack.

In [ ]:
!pip install -r requirements.txt --upgrade-strategy only-if-needed

## 2. Prepare the structured dataset

The source dialogues are split before creating per-assistant-turn examples to prevent turn leakage between train and test.

In [ ]:
!python scripts/prepare_data.py

## 3. Fine-tune the model

The default is Qwen2.5-0.5B-Instruct. Start without 4-bit quantization; reduce the batch size to 1 if memory is tight. This step downloads the model and creates a new adapter under `models/sora-qwen-lora/final`.

In [ ]:
!python scripts/train.py --epochs 5 --batch_size 2 --gradient_accumulation_steps 2 --lr 2e-4

## 4. Review a consistent prompt suite

Please read the generated answers manually. A successful script run alone does not establish response quality.

In [ ]:
!python scripts/evaluate.py

## 5. Start the chat interface

The public Gradio share link is accessible to anyone who has it while active. Add `--no-share` if you prefer not to create a public tunnel. Edge TTS requires an internet connection.

In [ ]:
!python scripts/chat.py --gradio --voice